# Notebook 11 — Protocolo Experimental con Participantes Humanos

**Proyecto:** Modelo Cognitivo Artificial para la Replicación de la Actividad Neurofisiológica de Percepción de Profundidad  
**Autor:** Jesús Goenaga Peña  
**Fase:** 5 — Validación con participantes humanos

---

## Estructura del protocolo (duración estimada: 60 minutos)

| Fase | Instrumento | Duración |
|------|-------------|----------|
| 1 | Consentimiento informado + firma electrónica | 5 min |
| 2 | Instrumento AdHoc (datos sociodemográficos y salud) | 10 min |
| 3 | BCSE — Guía de administración + registro de puntuaciones | 10 min |
| 4 | Prueba de salud visual | 10 min |
| 5 | 190 tareas de percepción de profundidad | 30 min |

## Instrucciones para el investigador

- Ejecutar las celdas **en orden** antes de iniciar la sesión con el participante.
- **Celda 1:** montar Drive y configurar rutas.
- **Celda 2:** registrar ID del participante (solo tú ves el mapeo ID↔nombre).
- A partir de la **Celda 3**, el participante puede ver la pantalla.
- Al finalizar cada fase, el investigador avanza manualmente a la siguiente celda.
- Los datos se guardan automáticamente en Drive al completar cada fase.

## Confidencialidad

Los datos se almacenan **únicamente con el ID anonimizado** del participante. El mapeo nombre↔ID se guarda cifrado en `data/private/`. Las respuestas de las tareas nunca contienen el nombre real.

---

## Celda 1 — Configuración inicial (solo investigador)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, csv, hashlib, datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import cv2
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

BASE_DIR     = '/content/drive/MyDrive/cognitive-depth-model'
SPLITS_DIR   = os.path.join(BASE_DIR, 'data', 'splits')
DATA_DIR     = os.path.join(BASE_DIR, 'data', 'participants')
PRIVATE_DIR  = os.path.join(BASE_DIR, 'data', 'private')
RESULTS_DIR  = os.path.join(BASE_DIR, 'results', 'human_responses')

for d in [DATA_DIR, PRIVATE_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

CSV_VALIDATION = os.path.join(SPLITS_DIR, 'validation_set_final.csv')
df_val = pd.read_csv(CSV_VALIDATION)

print('✓ Drive montado y rutas configuradas.')
print(f'  Set de validación: {len(df_val)} tareas')
print()
print('Participantes ya registrados:')
existing = [f for f in os.listdir(DATA_DIR) if f.endswith('.json')]
if existing:
    for e in sorted(existing):
        print(f'  {e}')
else:
    print('  (ninguno aún)')

## Celda 2 — Registro del participante (solo investigador, no compartir pantalla)

In [ ]:
# ─── SOLO INVESTIGADOR ─────────────────────────────────────────────────────
# Asigna el ID del participante. El nombre real NUNCA aparecerá en los datos.
# Usa P001, P002, P003 en orden.

PARTICIPANT_ID = 'P001'   # ← CAMBIAR PARA CADA PARTICIPANTE

FECHA_SESION  = datetime.datetime.now().strftime('%Y-%m-%d')
HORA_INICIO   = datetime.datetime.now().strftime('%H:%M:%S')
SESSION_CODE  = hashlib.md5(f'{PARTICIPANT_ID}{FECHA_SESION}{HORA_INICIO}'.encode()).hexdigest()[:8].upper()

# Verificar que no se repita el ID
id_file = os.path.join(DATA_DIR, f'{PARTICIPANT_ID}_session.json')
if os.path.exists(id_file):
    print(f'⚠ Ya existe sesión para {PARTICIPANT_ID}. Verifica el ID antes de continuar.')
else:
    print(f'✓ ID asignado: {PARTICIPANT_ID}')
    print(f'  Fecha: {FECHA_SESION} | Hora: {HORA_INICIO}')
    print(f'  Código de sesión: {SESSION_CODE}')
    print()
    print('Ahora puedes compartir pantalla con el participante y ejecutar la Celda 3.')

# Inicializar contenedor de datos de la sesión
session_data = {
    'participant_id':  PARTICIPANT_ID,
    'session_code':    SESSION_CODE,
    'fecha':           FECHA_SESION,
    'hora_inicio':     HORA_INICIO,
    'consentimiento':  {},
    'adhoc':           {},
    'bcse':            {},
    'salud_visual':    {},
    'tareas':          []
}

---
## FASE 1 — Consentimiento Informado
### Celda 3 — Presentación y firma electrónica del consentimiento

In [ ]:
display(HTML(f'''
<div style="font-family: Arial, sans-serif; max-width: 800px; margin: 0 auto; padding: 20px;">
<div style="background: #1a5276; color: white; padding: 15px; border-radius: 8px 8px 0 0; text-align: center;">
  <h2 style="margin:0;">CONSENTIMIENTO INFORMADO</h2>
  <p style="margin:5px 0 0;">Grupo de Investigación: Sistemas Cognitivos Artificiales</p>
</div>
<div style="border: 2px solid #1a5276; border-top: none; padding: 25px; border-radius: 0 0 8px 8px; background: #fdfefe;">

<p style="text-align:right;">Manizales, {FECHA_SESION}</p>

<p><strong>Investigación:</strong> Modelo Cognitivo Artificial para la Replicación de la Actividad Neurofisiológica de Percepción de Profundidad</p>
<p><strong>Investigador principal:</strong> Jesús Goenaga Peña — Doctorando en Ciencias Cognitivas, Universidad Autónoma de Manizales</p>
<p><strong>Director:</strong> Luis Fernando Castillo Ossa (PhD)</p>

<h3>¿En qué consiste su participación?</h3>
<p>Si acepta participar, se le pedirá completar las siguientes actividades en una sola sesión de aproximadamente <strong>60 minutos</strong>:</p>
<ol>
  <li>Diligenciar un formulario con datos sociodemográficos y auto-reporte de salud (Instrumento AdHoc).</li>
  <li>Participar en la aplicación del Test Breve de Evaluación del Estado Cognitivo (BCSE).</li>
  <li>Realizar pruebas breves de salud visual (agudeza visual y visión binocular).</li>
  <li>Completar 190 tareas de percepción de profundidad visual, respondiendo si un objeto (A) aparece más cercano o más lejano que otro (B) en cada imagen.</li>
</ol>

<h3>Información importante sobre su participación</h3>
<ul>
  <li>Su participación es completamente <strong>libre y voluntaria</strong>. Puede retirarse en cualquier momento sin penalización.</li>
  <li>Se registrarán automáticamente sus respuestas y tiempos de reacción. <strong>No</strong> se realizará registro fotográfico ni de video.</li>
  <li>Sus datos serán almacenados con un <strong>código anónimo</strong>. Su nombre nunca aparecerá en los archivos de datos.</li>
  <li>La información recopilada se usará exclusivamente para fines de investigación científica y podrá ser publicada en revistas académicas de forma anonimizada.</li>
  <li>Recibirá una <strong>compensación económica razonable</strong> por su tiempo, sin que esto constituya presión indebida.</li>
  <li>No existen riesgos físicos conocidos. Algunas personas pueden experimentar ligero malestar visual tras el uso prolongado de pantallas; en ese caso, puede solicitar una pausa o detener la sesión.</li>
  <li>Para cualquier duda posterior, puede contactar al investigador: jesus.goenagap@autonoma.edu.co</li>
</ul>

<h3>Confidencialidad y protección de datos</h3>
<p>Sus datos personales serán tratados conforme a la Ley 1581 de 2012 (Habeas Data, Colombia). Los datos se almacenan en servidores protegidos con acceso restringido al equipo investigador. El mapeo entre su nombre y su código de participante se guarda cifrado y separado de los datos de respuesta.</p>

<div style="background: #eaf2ff; padding: 15px; border-radius: 6px; border-left: 4px solid #1a5276; margin-top: 20px;">
  <p style="margin:0;"><strong>Código de sesión (para sus registros):</strong> {SESSION_CODE}</p>
  <p style="margin:5px 0 0; font-size:0.9em;">Conserve este código si desea solicitar la eliminación de sus datos en el futuro.</p>
</div>
</div>
</div>
'''))

print()
print('─' * 60)
print('FIRMA ELECTRÓNICA')
print('─' * 60)
print('Por favor, lea el documento anterior y complete los campos:')
print()

nombre_widget = widgets.Text(
    description='Nombre completo:',
    layout=widgets.Layout(width='500px'),
    style={'description_width': '150px'}
)
id_widget = widgets.Text(
    description='No. documento:',
    layout=widgets.Layout(width='400px'),
    style={'description_width': '150px'}
)
acepta_widget = widgets.Checkbox(
    value=False,
    description='He leído y comprendido el consentimiento informado y acepto participar voluntariamente.',
    layout=widgets.Layout(width='700px')
)
btn_firmar = widgets.Button(
    description='Confirmar firma electrónica',
    button_style='success',
    layout=widgets.Layout(width='280px', height='40px')
)
out_firma = widgets.Output()

def on_firmar(b):
    with out_firma:
        clear_output()
        if not acepta_widget.value:
            print('⚠ Debe marcar la casilla de aceptación antes de continuar.')
            return
        if not nombre_widget.value.strip():
            print('⚠ Por favor ingrese su nombre completo.')
            return
        if not id_widget.value.strip():
            print('⚠ Por favor ingrese su número de documento.')
            return
        timestamp_firma = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        session_data['consentimiento'] = {
            'nombre_hash':  hashlib.sha256(nombre_widget.value.strip().encode()).hexdigest(),
            'doc_hash':     hashlib.sha256(id_widget.value.strip().encode()).hexdigest(),
            'acepto':       True,
            'timestamp':    timestamp_firma,
            'session_code': SESSION_CODE
        }
        # Guardar mapeo privado (nombre real → ID) cifrado como hash
        mapeo_privado = {
            'participant_id':  PARTICIPANT_ID,
            'nombre_original': nombre_widget.value.strip(),
            'documento':       id_widget.value.strip(),
            'timestamp':       timestamp_firma,
            'session_code':    SESSION_CODE
        }
        ruta_privada = os.path.join(PRIVATE_DIR, f'{PARTICIPANT_ID}_identity.json')
        with open(ruta_privada, 'w') as f:
            json.dump(mapeo_privado, f, ensure_ascii=False, indent=2)
        print(f'✓ Consentimiento firmado electrónicamente.')
        print(f'  Nombre: {nombre_widget.value.strip()}')
        print(f'  Timestamp: {timestamp_firma}')
        print(f'  Código de sesión: {SESSION_CODE}')
        print()
        print('El investigador puede ahora ejecutar la Celda 4 (Instrumento AdHoc).')

btn_firmar.on_click(on_firmar)
display(nombre_widget, id_widget, acepta_widget, btn_firmar, out_firma)

---
## FASE 2 — Instrumento AdHoc
### Celda 4 — Datos sociodemográficos y salud

In [ ]:
display(HTML('<h2 style="font-family:Arial; color:#1a5276; border-bottom:2px solid #1a5276; padding-bottom:8px;">📋 INSTRUMENTO AdHoc</h2><p style="font-family:Arial;">Complete los siguientes campos. Sus respuestas son confidenciales.</p>'))

# ─── Datos sociodemográficos ───────────────────────────────────
edad_w = widgets.BoundedIntText(description='Edad:', min=18, max=99, value=30,
    layout=widgets.Layout(width='250px'), style={'description_width':'120px'})
genero_w = widgets.RadioButtons(description='Género:', options=['Masculino','Femenino','Otro'],
    layout=widgets.Layout(width='350px'), style={'description_width':'120px'})
educacion_w = widgets.Select(description='Nivel educativo:',
    options=['Sin educación formal','Primaria','Bachillerato','Técnico/Tecnológico',
             'Carrera profesional','Especialización','Maestría','Doctorado'],
    layout=widgets.Layout(width='450px'), style={'description_width':'150px'})
ocupacion_w = widgets.Select(description='Ocupación:',
    options=['Sin empleo','Estudiante','Trabajador independiente','Trabajador dependiente','Otro'],
    layout=widgets.Layout(width='450px'), style={'description_width':'150px'})
estado_civil_w = widgets.Select(description='Estado civil:',
    options=['Soltero/a','Casado/a','Conviviente','Separado/a','Divorciado/a','Viudo/a'],
    layout=widgets.Layout(width='350px'), style={'description_width':'150px'})

# ─── Salud visual ──────────────────────────────────────────────
prob_visual_w = widgets.RadioButtons(description='¿Problema visual diagnosticado?',
    options=['Sí','No'], layout=widgets.Layout(width='400px'), style={'description_width':'270px'})
tipo_prob_visual_w = widgets.SelectMultiple(description='Tipo de problema:',
    options=['Miopía','Hipermetropía','Astigmatismo','Presbicia','Otro'],
    layout=widgets.Layout(width='400px'), style={'description_width':'150px'})
correccion_w = widgets.RadioButtons(description='Corrección visual:',
    options=['Gafas','Lentes de contacto','Ninguna'],
    layout=widgets.Layout(width='350px'), style={'description_width':'150px'})
examen_w = widgets.RadioButtons(description='Frecuencia de exámenes de vista:',
    options=['Regularmente (cada 1-2 años)','De vez en cuando (>2 años)','Nunca'],
    layout=widgets.Layout(width='500px'), style={'description_width':'250px'})

# ─── Salud neurocognitiva ──────────────────────────────────────
trastorno_cog_w = widgets.RadioButtons(description='¿Trastorno neurocognitivo diagnosticado?',
    options=['Sí','No'], layout=widgets.Layout(width='450px'), style={'description_width':'310px'})
cambios_cog_w = widgets.SelectMultiple(description='Cambios recientes:',
    options=['Cambios en memoria reciente','Dificultad para concentrarse',
             'Dificultad para resolver problemas','Ninguno de los anteriores'],
    layout=widgets.Layout(width='500px'), style={'description_width':'150px'})

# ─── Salud psiquiátrica ────────────────────────────────────────
trastorno_psiq_w = widgets.RadioButtons(description='¿Trastorno psiquiátrico diagnosticado?',
    options=['Sí','No'], layout=widgets.Layout(width='450px'), style={'description_width':'310px'})
sintomas_psiq_w = widgets.RadioButtons(description='¿Síntomas psiquiátricos recientes?',
    options=['Sí','No'], layout=widgets.Layout(width='400px'), style={'description_width':'250px'})
tratamiento_psiq_w = widgets.RadioButtons(description='¿Tratamiento psiquiátrico último año?',
    options=['Sí','No'], layout=widgets.Layout(width='400px'), style={'description_width':'260px'})

btn_adhoc = widgets.Button(description='Guardar respuestas AdHoc',
    button_style='primary', layout=widgets.Layout(width='260px', height='40px'))
out_adhoc = widgets.Output()

def on_adhoc(b):
    with out_adhoc:
        clear_output()
        session_data['adhoc'] = {
            'edad':                  edad_w.value,
            'genero':                genero_w.value,
            'nivel_educativo':       educacion_w.value,
            'ocupacion':             ocupacion_w.value,
            'estado_civil':          estado_civil_w.value,
            'problema_visual':       prob_visual_w.value,
            'tipo_problema_visual':  list(tipo_prob_visual_w.value),
            'correccion_visual':     correccion_w.value,
            'frecuencia_examen':     examen_w.value,
            'trastorno_neurocog':    trastorno_cog_w.value,
            'cambios_cognitivos':    list(cambios_cog_w.value),
            'trastorno_psiquiatrico':trastorno_psiq_w.value,
            'sintomas_psiquiatricos':sintomas_psiq_w.value,
            'tratamiento_psiquiatrico': tratamiento_psiq_w.value,
            'timestamp':             datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        }
        print('✓ Respuestas AdHoc guardadas.')
        print(f'  Participante {PARTICIPANT_ID} | Edad: {edad_w.value} | Género: {genero_w.value}')
        print()
        print('El investigador puede ejecutar la Celda 5 (BCSE).')

btn_adhoc.on_click(on_adhoc)

display(HTML('<h3 style="font-family:Arial;">Datos sociodemográficos</h3>'))
display(edad_w, genero_w, educacion_w, ocupacion_w, estado_civil_w)
display(HTML('<h3 style="font-family:Arial;">Salud visual</h3>'))
display(prob_visual_w, tipo_prob_visual_w, correccion_w, examen_w)
display(HTML('<h3 style="font-family:Arial;">Salud neurocognitiva</h3>'))
display(trastorno_cog_w, cambios_cog_w)
display(HTML('<h3 style="font-family:Arial;">Salud psiquiátrica</h3>'))
display(trastorno_psiq_w, sintomas_psiq_w, tratamiento_psiq_w)
display(btn_adhoc, out_adhoc)

---
## FASE 3 — BCSE (Test Breve para la Evaluación del Estado Cognitivo)
### Celda 5 — Guía de administración y registro de puntuaciones

> **Instrucción para el investigador:** Use el cuaderno de estímulos físico de Pearson. El participante responde verbalmente. Usted registra las puntuaciones en esta celda. La interfaz calcula automáticamente los puntajes ponderados.

In [ ]:
display(HTML('''
<div style="font-family:Arial; background:#eaf2ff; padding:15px; border-radius:8px; border-left:4px solid #1a5276; margin-bottom:15px;">
<strong>⚠ Instrucción para el investigador:</strong> Los ítems se administran verbalmente con el cuaderno de estímulos físico (Pearson).
Registre únicamente las puntuaciones directas en los campos a continuación. El sistema calculará los puntajes ponderados automáticamente.
</div>
<h2 style="color:#1a5276;">📊 BCSE — Registro de Puntuaciones</h2>
'''))

# ─── Orientación (ítems 1-5, máx=5) ──────────────────────────
display(HTML('<h3 style="font-family:Arial; color:#2471a3;">1. Orientación (ítems 1-5)</h3><p style="font-family:Arial; font-size:0.9em;">Pregunte: año, mes, día de la semana, día del mes, presidente del gobierno. 1 punto por respuesta correcta.</p>'))
orient_w = widgets.BoundedIntText(description='Puntuación directa (0-5):',
    min=0, max=5, value=0, layout=widgets.Layout(width='300px'), style={'description_width':'200px'})
display(orient_w)

# ─── Estimación temporal (ítem 6) ─────────────────────────────
display(HTML('<h3 style="font-family:Arial; color:#2471a3;">2. Estimación temporal (ítem 6)</h3><p style="font-family:Arial; font-size:0.9em;">Diferencia en minutos entre la hora dicha y la hora real.</p>'))
et_dif_w = widgets.BoundedIntText(description='Diferencia en minutos:',
    min=0, max=720, value=0, layout=widgets.Layout(width='300px'), style={'description_width':'200px'})
display(et_dif_w)

# ─── Control mental: Tiempo (ítem 7-8) ────────────────────────
display(HTML('<h3 style="font-family:Arial; color:#2471a3;">3. Control mental</h3><p style="font-family:Arial; font-size:0.9em;">Registre tiempo en segundos y número de errores.</p>'))
cm_tiempo_w = widgets.BoundedIntText(description='Tiempo (segundos):',
    min=0, max=300, value=0, layout=widgets.Layout(width='280px'), style={'description_width':'180px'})
cm_errores_w = widgets.BoundedIntText(description='Errores:',
    min=0, max=20, value=0, layout=widgets.Layout(width='250px'), style={'description_width':'150px'})
display(cm_tiempo_w, cm_errores_w)

# ─── Dibujo del reloj (ítem 9, máx=15) ───────────────────────
display(HTML('<h3 style="font-family:Arial; color:#2471a3;">4. Dibujo del reloj (ítem 9)</h3><p style="font-family:Arial; font-size:0.9em;">Puntuación directa total sobre el dibujo (máx=15).</p>'))
reloj_w = widgets.BoundedIntText(description='Puntuación directa (0-15):',
    min=0, max=15, value=0, layout=widgets.Layout(width='300px'), style={'description_width':'220px'})
display(reloj_w)

# ─── Recuerdo incidental (ítem 10, máx=4) ─────────────────────
display(HTML('<h3 style="font-family:Arial; color:#2471a3;">5. Recuerdo incidental (ítem 10)</h3><p style="font-family:Arial; font-size:0.9em;">Palabras recordadas de: silla, gafas, pájaro, taza.</p>'))
recuerdo_w = widgets.BoundedIntText(description='Palabras recordadas (0-4):',
    min=0, max=4, value=0, layout=widgets.Layout(width='300px'), style={'description_width':'220px'})
display(recuerdo_w)

# ─── Inhibición (ítem 11) ──────────────────────────────────────
display(HTML('<h3 style="font-family:Arial; color:#2471a3;">6. Inhibición (ítem 11)</h3>'))
inhib_tiempo_w = widgets.BoundedIntText(description='Tiempo (segundos):',
    min=0, max=300, value=0, layout=widgets.Layout(width='280px'), style={'description_width':'180px'})
inhib_omis_w = widgets.BoundedIntText(description='Omisiones:',
    min=0, max=24, value=0, layout=widgets.Layout(width='250px'), style={'description_width':'150px'})
inhib_comis_w = widgets.BoundedIntText(description='Comisiones:',
    min=0, max=24, value=0, layout=widgets.Layout(width='250px'), style={'description_width':'150px'})
display(inhib_tiempo_w, inhib_omis_w, inhib_comis_w)

# ─── Producción verbal (ítem 12) ──────────────────────────────
display(HTML('<h3 style="font-family:Arial; color:#2471a3;">7. Producción verbal (ítem 12)</h3><p style="font-family:Arial; font-size:0.9em;">Número de palabras en 30 segundos.</p>'))
pv_w = widgets.BoundedIntText(description='Palabras producidas:',
    min=0, max=50, value=0, layout=widgets.Layout(width='280px'), style={'description_width':'200px'})
display(pv_w)

btn_bcse = widgets.Button(description='Calcular y guardar BCSE',
    button_style='primary', layout=widgets.Layout(width='260px', height='40px'))
out_bcse = widgets.Output()

def convertir_ponderada_orientacion(directa):
    if directa <= 3: return 0
    elif directa == 4: return 5
    else: return 8

def convertir_ponderada_et(dif):
    if dif > 55: return 0
    elif dif >= 31: return 1
    elif dif >= 19: return 2
    elif dif >= 10: return 3
    else: return 4

def convertir_ponderada_cm_tiempo(seg):
    if seg > 75: return 0
    elif seg >= 56: return 1
    elif seg >= 43: return 2
    elif seg >= 31: return 3
    else: return 4

def convertir_ponderada_cm_errores(err):
    if err > 3: return 0
    elif err >= 2: return 2
    elif err == 1: return 4
    else: return 8

def convertir_ponderada_reloj(directa):
    if directa <= 6: return 0
    elif directa <= 8: return 1
    elif directa == 9: return 2
    elif directa <= 11: return 3
    else: return 4

def convertir_ponderada_recuerdo(directa):
    if directa == 0: return 0
    elif directa == 1: return 2
    elif directa == 2: return 4
    else: return 8

def convertir_ponderada_inhib_tiempo(seg):
    if seg > 48: return 0
    elif seg >= 38: return 1
    elif seg >= 30: return 2
    elif seg >= 23: return 3
    else: return 4

def convertir_ponderada_inhib_omis(omis):
    if omis >= 1: return 0
    else: return 4

def convertir_ponderada_inhib_comis(comis):
    tabla = {range(8,25):0, range(7,8):1, range(6,7):2, range(5,6):3,
             range(4,5):4, range(3,4):5, range(2,3):6, range(1,2):7, range(0,1):8}
    for rango, pts in tabla.items():
        if comis in rango: return pts
    return 0

def convertir_ponderada_pv(palabras):
    if palabras <= 6: return 0
    elif palabras == 7: return 1
    elif palabras == 8: return 2
    elif palabras == 9: return 4
    else: return 6

def on_bcse(b):
    with out_bcse:
        clear_output()
        pond_orient = convertir_ponderada_orientacion(orient_w.value)
        pond_et     = convertir_ponderada_et(et_dif_w.value)
        pond_cm_t   = convertir_ponderada_cm_tiempo(cm_tiempo_w.value)
        pond_cm_e   = convertir_ponderada_cm_errores(cm_errores_w.value)
        pond_reloj  = convertir_ponderada_reloj(reloj_w.value)
        pond_recuerdo = convertir_ponderada_recuerdo(recuerdo_w.value)
        pond_inh_t  = convertir_ponderada_inhib_tiempo(inhib_tiempo_w.value)
        pond_inh_o  = convertir_ponderada_inhib_omis(inhib_omis_w.value)
        pond_inh_c  = convertir_ponderada_inhib_comis(inhib_comis_w.value)
        pond_pv     = convertir_ponderada_pv(pv_w.value)
        total_bcse  = pond_orient + pond_et + pond_cm_t + pond_cm_e + pond_reloj + pond_recuerdo + pond_inh_t + pond_inh_o + pond_inh_c + pond_pv
        session_data['bcse'] = {
            'orientacion_directa':  orient_w.value,
            'orientacion_ponderada': pond_orient,
            'et_diferencia_min':    et_dif_w.value,
            'et_ponderada':         pond_et,
            'cm_tiempo_seg':        cm_tiempo_w.value,
            'cm_tiempo_ponderada':  pond_cm_t,
            'cm_errores':           cm_errores_w.value,
            'cm_errores_ponderada': pond_cm_e,
            'reloj_directa':        reloj_w.value,
            'reloj_ponderada':      pond_reloj,
            'recuerdo_directa':     recuerdo_w.value,
            'recuerdo_ponderada':   pond_recuerdo,
            'inhibicion_tiempo_seg':inhib_tiempo_w.value,
            'inhibicion_tiempo_pond':pond_inh_t,
            'inhibicion_omisiones': inhib_omis_w.value,
            'inhibicion_omis_pond': pond_inh_o,
            'inhibicion_comisiones':inhib_comis_w.value,
            'inhibicion_comis_pond':pond_inh_c,
            'produccion_verbal_directa': pv_w.value,
            'produccion_verbal_ponderada': pond_pv,
            'total_bcse':           total_bcse,
            'timestamp':            datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        }
        print(f'✓ BCSE calculado y guardado.')
        print(f'  Puntuación total BCSE: {total_bcse}/58')
        print()
        if total_bcse >= 47:
            print('  → Clasificación: NORMAL (puede continuar con el protocolo)')
        elif total_bcse >= 40:
            print('  → Clasificación: NORMAL-BAJO / LÍMITE (documentar y continuar con precaución)')
        else:
            print('  → Clasificación: BAJO / MUY BAJO (considerar criterio de exclusión)')
        print()
        print('Ejecute la Celda 6 para continuar con la Prueba de Salud Visual.')

btn_bcse.on_click(on_bcse)
display(btn_bcse, out_bcse)

---
## FASE 4 — Prueba de Salud Visual
### Celda 6 — Agudeza visual, visión binocular e historia autoinformada

In [ ]:
display(HTML('<h2 style="font-family:Arial; color:#1a5276; border-bottom:2px solid #1a5276; padding-bottom:8px;">👁 PRUEBA DE SALUD VISUAL</h2>'))

# ─── Sección 1: Agudeza visual ────────────────────────────────
display(HTML('<h3 style="font-family:Arial; color:#2471a3;">Sección 1. Agudeza visual (cartilla Snellen a 3 metros)</h3><p style="font-family:Arial; font-size:0.9em;">Evaluar un ojo a la vez. Registrar el nivel más alto alcanzado sin errores.</p>'))
ojo_izq_w = widgets.Select(description='Ojo izquierdo:',
    options=['20/20','20/25','20/30','20/40','Peor de 20/40','No evaluable'],
    layout=widgets.Layout(width='350px'), style={'description_width':'130px'})
ojo_der_w = widgets.Select(description='Ojo derecho:',
    options=['20/20','20/25','20/30','20/40','Peor de 20/40','No evaluable'],
    layout=widgets.Layout(width='350px'), style={'description_width':'130px'})
correccion_usada_w = widgets.RadioButtons(description='¿Usó corrección?',
    options=['Sí (gafas)','Sí (lentes de contacto)','No'],
    layout=widgets.Layout(width='350px'), style={'description_width':'140px'})
display(ojo_izq_w, ojo_der_w, correccion_usada_w)

# ─── Sección 2: Visión binocular ──────────────────────────────
display(HTML('<h3 style="font-family:Arial; color:#2471a3;">Sección 2. Visión binocular y disparidad estereoscópica</h3><p style="font-family:Arial; font-size:0.9em;">Presente 5 imágenes anaglifas con gafas rojo/azul. Registre los ítems respondidos correctamente.</p>'))
estereop_w = widgets.BoundedIntText(description='Ítems correctos (0-5):',
    min=0, max=5, value=0, layout=widgets.Layout(width='280px'), style={'description_width':'200px'})
display(estereop_w)

# ─── Sección 3: Historia visual autoinformada ─────────────────
display(HTML('<h3 style="font-family:Arial; color:#2471a3;">Sección 3. Historia visual autoinformada</h3>'))
hist_preguntas = [
    '¿Usa actualmente gafas o lentes de contacto?',
    '¿Le han diagnosticado miopía, astigmatismo, hipermetropía o estrabismo?',
    '¿Ha sido sometido a cirugía ocular (láser, cataratas, etc.)?',
    '¿Ha tenido visión doble o dificultad para enfocar en los últimos 6 meses?',
    '¿Tiene antecedentes de enfermedades neurológicas que afecten la visión?',
    '¿Le cuesta seguir objetos en movimiento o detectar diferencias de profundidad?'
]
hist_widgets = []
for preg in hist_preguntas:
    w = widgets.RadioButtons(description=preg,
        options=['Sí','No','No sabe'],
        layout=widgets.Layout(width='700px'), style={'description_width':'500px'})
    hist_widgets.append(w)
    display(w)

btn_visual = widgets.Button(description='Guardar y evaluar salud visual',
    button_style='primary', layout=widgets.Layout(width='280px', height='40px'))
out_visual = widgets.Output()

def clasificar_agudeza(nivel):
    adecuados = ['20/20','20/25']
    sospechosos = ['20/30','20/40']
    if nivel in adecuados: return 'adecuada'
    elif nivel in sospechosos: return 'sospechosa'
    else: return 'limitada'

def on_visual(b):
    with out_visual:
        clear_output()
        cls_izq = clasificar_agudeza(ojo_izq_w.value)
        cls_der = clasificar_agudeza(ojo_der_w.value)
        n_afirmativos = sum(1 for w in hist_widgets if w.value == 'Sí')
        # Criterios de exclusión
        excluir_agudeza = (cls_izq == 'limitada' and cls_der == 'limitada')
        excluir_estereop = (estereop_w.value < 3)
        revision_adicional = (n_afirmativos >= 3)
        session_data['salud_visual'] = {
            'ojo_izquierdo':        ojo_izq_w.value,
            'ojo_derecho':          ojo_der_w.value,
            'clase_ojo_izq':        cls_izq,
            'clase_ojo_der':        cls_der,
            'correccion_usada':     correccion_usada_w.value,
            'estereopsia_correctos':estereop_w.value,
            'historia_autoinformada': [w.value for w in hist_widgets],
            'n_afirmativos_historia': n_afirmativos,
            'criterio_exclusion_agudeza':   excluir_agudeza,
            'criterio_exclusion_estereop':  excluir_estereop,
            'requiere_revision_adicional':  revision_adicional,
            'timestamp': datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        }
        print('✓ Prueba de salud visual guardada.')
        print(f'  Agudeza OI: {ojo_izq_w.value} ({cls_izq}) | OD: {ojo_der_w.value} ({cls_der})')
        print(f'  Estereopsia: {estereop_w.value}/5 ítems correctos')
        print(f'  Historia autoinformada: {n_afirmativos} respuestas afirmativas')
        print()
        if excluir_agudeza:
            print('  ⛔ CRITERIO DE EXCLUSIÓN: Agudeza visual limitada en ambos ojos.')
        elif excluir_estereop:
            print('  ⛔ CRITERIO DE EXCLUSIÓN: No percibe profundidad estereoscópica (< 3/5).')
        elif revision_adicional:
            print('  ⚠ Participación condicionada: revisar historia visual con experto.')
        else:
            print('  ✓ Participante apto para las tareas experimentales.')
        print()
        print('Si es apto, ejecute la Celda 7 para iniciar las 190 tareas.')

btn_visual.on_click(on_visual)
display(btn_visual, out_visual)

---
## FASE 5 — 190 Tareas de Percepción de Profundidad
### Celda 7 — Instrucciones y configuración

In [ ]:
display(HTML(f'''
<div style="font-family:Arial; max-width:700px; margin:0 auto;">
<div style="background:#1a5276; color:white; padding:15px; border-radius:8px 8px 0 0; text-align:center;">
  <h2 style="margin:0;">TAREAS DE PERCEPCIÓN DE PROFUNDIDAD</h2>
</div>
<div style="border:2px solid #1a5276; border-top:none; padding:25px; border-radius:0 0 8px 8px; background:#fdfefe;">
<h3>Instrucciones</h3>
<p>A continuación verá una serie de <strong>190 imágenes</strong>. En cada imagen habrá dos regiones marcadas:</p>
<ul>
  <li><span style="color:#e74c3c; font-weight:bold;">■ Región A</span> — marcada con un recuadro rojo</li>
  <li><span style="color:#2980b9; font-weight:bold;">■ Región B</span> — marcada con un recuadro azul</li>
</ul>
<p>Su tarea es indicar si el objeto en la <strong>Región A</strong> le parece:</p>
<ul>
  <li><strong>Más CERCANO</strong> que el objeto en la Región B</li>
  <li><strong>Más LEJANO</strong> que el objeto en la Región B</li>
</ul>
<p>Responda de acuerdo a su percepción espontánea. No hay respuestas correctas o incorrectas desde su punto de vista perceptual.</p>
<p>Puede tomarse un descanso en cualquier momento avisando al investigador.</p>
<div style="background:#eaf2ff; padding:10px; border-radius:6px; margin-top:15px;">
  <strong>Participante:</strong> {PARTICIPANT_ID} &nbsp;|&nbsp;
  <strong>Total tareas:</strong> 190 &nbsp;|&nbsp;
  <strong>Tiempo estimado:</strong> 30 minutos
</div>
</div>
</div>
'''))

print()
entendio_w = widgets.Checkbox(value=False,
    description='El participante confirma que entendió las instrucciones.',
    layout=widgets.Layout(width='500px'))
btn_iniciar = widgets.Button(description='▶ Iniciar tareas',
    button_style='success', layout=widgets.Layout(width='200px', height='50px'))
out_inicio = widgets.Output()

tareas_lista = df_val.to_dict('records')
respuestas_tareas = []
tarea_actual = [0]  # mutable para uso en callback

# Widgets reutilizables para las tareas
out_imagen  = widgets.Output(layout=widgets.Layout(width='100%'))
out_info    = widgets.Output()
btn_cercano = widgets.Button(description='🔴 A está MÁS CERCANO',
    button_style='danger', layout=widgets.Layout(width='280px', height='55px'))
btn_lejano  = widgets.Button(description='🔵 A está MÁS LEJANO',
    button_style='info',   layout=widgets.Layout(width='280px', height='55px'))
out_feedback = widgets.Output()
progress_bar = widgets.IntProgress(value=0, min=0, max=len(tareas_lista),
    description='Progreso:', layout=widgets.Layout(width='600px'))
lbl_progreso = widgets.Label(value=f'0 / {len(tareas_lista)} tareas completadas')

t_inicio_tarea = [None]

def mostrar_tarea(idx):
    if idx >= len(tareas_lista):
        with out_imagen:
            clear_output()
            print('✓ Todas las tareas completadas.')
        btn_cercano.disabled = True
        btn_lejano.disabled  = True
        return
    tarea = tareas_lista[idx]
    ruta_img = str(tarea.get('ruta_img_izq',''))
    t_inicio_tarea[0] = datetime.datetime.now()
    with out_imagen:
        clear_output(wait=True)
        try:
            img = cv2.imread(ruta_img)
            if img is None:
                print(f'⚠ Imagen no encontrada: {ruta_img}')
                return
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            fig, ax = plt.subplots(1, 1, figsize=(12, 4))
            ax.imshow(img_rgb)
            # Dibujar región A (rojo)
            rect_a = patches.Rectangle(
                (tarea['A_x'], tarea['A_y']), tarea['A_ancho'], tarea['A_alto'],
                linewidth=3, edgecolor='#e74c3c', facecolor='none')
            ax.add_patch(rect_a)
            ax.text(tarea['A_x']+6, tarea['A_y']+22, 'A', color='white', fontsize=16,
                    fontweight='bold', bbox=dict(facecolor='#e74c3c', alpha=0.9, pad=3))
            # Dibujar región B (azul)
            rect_b = patches.Rectangle(
                (tarea['B_x'], tarea['B_y']), tarea['B_ancho'], tarea['B_alto'],
                linewidth=3, edgecolor='#2980b9', facecolor='none')
            ax.add_patch(rect_b)
            ax.text(tarea['B_x']+6, tarea['B_y']+22, 'B', color='white', fontsize=16,
                    fontweight='bold', bbox=dict(facecolor='#2980b9', alpha=0.9, pad=3))
            ax.axis('off')
            ax.set_title(f'Tarea {idx+1} de {len(tareas_lista)} — ¿El objeto A está más cercano o más lejano que B?',
                        fontsize=13, fontweight='bold', pad=10)
            plt.tight_layout()
            plt.show()
        except Exception as ex:
            print(f'Error mostrando imagen: {ex}')
    with out_feedback:
        clear_output()

def registrar_respuesta(respuesta):
    idx   = tarea_actual[0]
    tarea = tareas_lista[idx]
    t_fin = datetime.datetime.now()
    t_ms  = int((t_fin - t_inicio_tarea[0]).total_seconds() * 1000) if t_inicio_tarea[0] else None
    respuestas_tareas.append({
        'participant_id':         PARTICIPANT_ID,
        'numero_tarea':           tarea['numero_tarea'],
        'imagen_id':              tarea['imagen_id'],
        'dataset':                tarea['dataset'],
        'tipo_tarea':             tarea['tipo_tarea'],
        'nivel_disparidad':       tarea['nivel_disparidad'],
        'presencia_distractores': tarea['presencia_distractores'],
        'ground_truth':           tarea['ground_truth'],
        'respuesta_humana':       respuesta,
        'acierto':                int(respuesta == tarea['ground_truth']),
        'tiempo_respuesta_ms':    t_ms,
        'timestamp':              t_fin.strftime('%Y-%m-%d %H:%M:%S')
    })
    tarea_actual[0] += 1
    n = tarea_actual[0]
    progress_bar.value = n
    lbl_progreso.value = f'{n} / {len(tareas_lista)} tareas completadas'
    # Guardado automático cada 25 tareas
    if n % 25 == 0 or n == len(tareas_lista):
        guardar_progreso()
    mostrar_tarea(n)

def guardar_progreso():
    if respuestas_tareas:
        df_resp = pd.DataFrame(respuestas_tareas)
        ruta = os.path.join(DATA_DIR, f'{PARTICIPANT_ID}_tareas.csv')
        df_resp.to_csv(ruta, index=False)

def on_cercano(b):
    btn_cercano.disabled = True
    btn_lejano.disabled  = True
    registrar_respuesta('mas_cercano')
    btn_cercano.disabled = False
    btn_lejano.disabled  = False

def on_lejano(b):
    btn_cercano.disabled = True
    btn_lejano.disabled  = True
    registrar_respuesta('mas_lejano')
    btn_cercano.disabled = False
    btn_lejano.disabled  = False

btn_cercano.on_click(on_cercano)
btn_lejano.on_click(on_lejano)

def on_iniciar(b):
    with out_inicio:
        clear_output()
        if not entendio_w.value:
            print('⚠ El participante debe confirmar que entendió las instrucciones.')
            return
    btn_iniciar.disabled = True
    display(progress_bar, lbl_progreso)
    display(out_imagen)
    display(widgets.HBox([btn_cercano, widgets.Label('  '), btn_lejano]))
    display(out_feedback)
    mostrar_tarea(0)

btn_iniciar.on_click(on_iniciar)
display(entendio_w, btn_iniciar, out_inicio)

---
## Celda 8 — Finalización y guardado de todos los datos

In [ ]:
# Ejecutar esta celda DESPUÉS de que el participante complete todas las tareas

hora_fin = datetime.datetime.now().strftime('%H:%M:%S')
session_data['hora_fin'] = hora_fin
session_data['tareas']   = respuestas_tareas

# ── 1. Guardar JSON completo de la sesión ─────────────────────
ruta_sesion = os.path.join(DATA_DIR, f'{PARTICIPANT_ID}_session.json')
with open(ruta_sesion, 'w', encoding='utf-8') as f:
    json.dump(session_data, f, ensure_ascii=False, indent=2)

# ── 2. Guardar CSV de respuestas de tareas ────────────────────
if respuestas_tareas:
    df_resp = pd.DataFrame(respuestas_tareas)
    ruta_tareas = os.path.join(DATA_DIR,    f'{PARTICIPANT_ID}_tareas.csv')
    ruta_result = os.path.join(RESULTS_DIR, f'{PARTICIPANT_ID}_tareas.csv')
    df_resp.to_csv(ruta_tareas,  index=False)
    df_resp.to_csv(ruta_result,  index=False)

# ── 3. Calcular métricas finales ───────────────────────────────
print('=' * 65)
print(f'SESIÓN COMPLETADA — {PARTICIPANT_ID}')
print('=' * 65)
print(f'Hora inicio: {session_data["hora_inicio"]} | Hora fin: {hora_fin}')
print()

if respuestas_tareas:
    df_r = pd.DataFrame(respuestas_tareas)
    n_completadas = len(df_r)
    acc_global    = df_r['acierto'].mean()
    tr_medio      = df_r['tiempo_respuesta_ms'].mean()
    print(f'Tareas completadas: {n_completadas}/{len(tareas_lista)}')
    print(f'Accuracy global:    {acc_global:.4f}  ({acc_global*100:.1f}%)')
    print(f'TR medio:           {tr_medio:.0f} ms')
    print()
    print('Accuracy por condición factorial:')
    print(df_r.groupby(['tipo_tarea','nivel_disparidad','presencia_distractores'])['acierto'].mean().round(3).to_string())

print()
print(f'BCSE total: {session_data["bcse"].get("total_bcse","N/A")}/58')
print()
print('Archivos guardados:')
print(f'  → {ruta_sesion}')
if respuestas_tareas:
    print(f'  → {ruta_tareas}')
    print(f'  → {ruta_result}')
print()
print('✓ Sesión finalizada correctamente. Gracias al participante.')